# Demo sintético — TrustGraph

Recorrido reproducible del proyecto: generación de datos, detección automática del esquema, construcción de Trust Cliente/Comercio, relaciones y grafo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from trustgraph import TrustGraphPipeline, generate_synthetic_transactions

## 1. Generar datos sintéticos

In [ ]:
df, metadata = generate_synthetic_transactions(
    n_clients=300, n_merchants=50, months=18, seed=42
)
print(df.shape)
df.head()

## 2. Detección automática de columnas y ejecución del pipeline

In [ ]:
pipeline = TrustGraphPipeline.from_yaml(ROOT / "config" / "default.yaml")
result = pipeline.run(df)
result.schema.mapping

## 3. Trust Cliente mensual

In [ ]:
cols = [
    "client_id", "month", "tx_count", "unique_merchants",
    "avg_relationship_strength", "trust_score", "trust_change_3m"
]
result.client_month[cols].tail(10)

## 4. Trust Comercio mensual

In [ ]:
cols = [
    "merchant_id", "month", "tx_count", "unique_clients",
    "repeat_customer_rate", "client_hhi", "trust_score", "trust_change_3m"
]
result.merchant_month[cols].tail(10)

## 5. Fortaleza Cliente–Comercio

In [ ]:
result.relationship_month[[
    "client_id", "merchant_id", "month", "tx_count",
    "relation_age_months", "relation_streak_months",
    "client_tx_share", "relationship_strength"
]].sort_values("relationship_strength", ascending=False).head(15)

## 6. Validar que relaciones persistentes son más fuertes

In [ ]:
rel = result.relationship_month
pd.Series({
    "relaciones_nuevas_1m": rel.loc[rel.relation_age_months == 1, "relationship_strength"].mean(),
    "relaciones_6m_o_mas": rel.loc[rel.relation_age_months >= 6, "relationship_strength"].mean(),
})

## 7. Validar deterioro sintético

In [ ]:
cm = result.client_month.copy()
cm["grupo"] = cm["client_id"].isin(metadata.deteriorating_clients).map({True:"Deterioro sintético", False:"Estable"})
series = cm.groupby(["month", "grupo"])["trust_score"].mean().unstack()
series.tail(8)

In [ ]:
ax = series.plot(figsize=(9,4), title="Trust Cliente promedio")
ax.set_ylabel("Trust Score")
ax.set_xlabel("Mes")
plt.tight_layout()
plt.show()

## 8. Grafo y comunidades

In [ ]:
print("Nodos:", result.graph.number_of_nodes())
print("Aristas:", result.graph.number_of_edges())
print("Comunidades cliente:", result.client_communities.community.nunique())
print("Comunidades comercio:", result.merchant_communities.community.nunique())
result.graph_metrics.sort_values("pagerank", ascending=False).head(10)

## Lectura del resultado

El score no intenta etiquetar fraude. Evalúa confianza comportamental observable. La validación sintética comprueba que: (1) relaciones persistentes reciben mayor fortaleza; y (2) entidades cuyo comportamiento se deteriora artificialmente muestran una caída mayor del Trust Score.